# Pneumonia (X-ray) — CNN Progress Report Notebook

This notebook implements a **CNN-only** pipeline aligned with the project progress report requirements:
- Dataset & preprocessing (resize/normalize, augmentation, stratified 80/10/10)
- Inputs/features (CNN representation learning; optional histogram equalization ablation)
- Model implementation (transfer learning, loss, optimizer, scheduler, early stopping)
- Baselines (majority class)
- Evaluation (accuracy, precision, recall, F1, ROC-AUC, confusion matrix)
- Ablations (head-only vs. fine-tune last block)

**Note:** Set `DATA_DIR` to your two-class folder with subfolders `NORMAL/` and `PNEUMONIA/`.

## 0) Setup & Config

In [1]:

# 0) Reproducibility & Config
import os, random, math, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
from PIL import ImageOps

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths & config
DATA_DIR = "chest_xray_binary"   # <<< CHANGE THIS TO YOUR DATA ROOT
OUT_DIR = "report_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

BATCH_SIZE = 32
IMG_SIZE = 224
VAL_SPLIT = 0.10
TEST_SPLIT = 0.10

EPOCHS_PHASE_A = 6            # head-only
EPOCHS_PHASE_B = 4            # unfreeze last block
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
PATIENCE = 3                  # early stopping on val F1
USE_HIST_EQUALIZE = False     # simple feature-engineering ablation
POSITIVE_CLASS_NAME = "PNEUMONIA"  # for AUROC / recall

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f'Device: {DEVICE}, Data dir exists: {os.path.isdir(DATA_DIR)}')

print('Setup done.')

Device: cpu, Data dir exists: False
Setup done.


## 1) Transforms (Preprocessing & Augmentation)

In [4]:

# 1) Transforms (preprocessing & augmentation)
class OptionalEqualize:
    """Apply histogram equalization and convert back to 3-channel."""
    def __init__(self, enabled: bool):
        self.enabled = enabled
    def __call__(self, img):
        if not self.enabled:
            return img
        gray = img.convert("L")
        eq = ImageOps.equalize(gray)
        return eq.convert("RGB")

tf_train = transforms.Compose([
    OptionalEqualize(USE_HIST_EQUALIZE),
    transforms.Grayscale(num_output_channels=3),           # make 3ch for ResNet backbones
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

tf_eval = transforms.Compose([
    OptionalEqualize(USE_HIST_EQUALIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

print('Transforms ready. USE_HIST_EQUALIZE =', USE_HIST_EQUALIZE)


Transforms ready. USE_HIST_EQUALIZE = False


## 2) Load Dataset & Stratified Split (80/10/10)

In [ ]:
# 2) Locate data & build train/test datasets from your folders
from pathlib import Path

DATA_ROOT = Path("chest_xray")        # <<< your root
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"

assert TRAIN_DIR.exists(), f"Missing {TRAIN_DIR}"
assert TEST_DIR.exists(),  f"Missing {TEST_DIR}"

# Build datasets (train uses augmentation; test uses eval transforms)
train_full_ds = datasets.ImageFolder(root=str(TRAIN_DIR), transform=tf_train)
test_ds       = datasets.ImageFolder(root=str(TEST_DIR),  transform=tf_eval)

class_to_idx = train_full_ds.class_to_idx   # {'NORMAL':0,'PNEUMONIA':1} (order depends on folder names)
idx_to_class = {v:k for k,v in class_to_idx.items()}

print("Class mapping:", class_to_idx)
print("Train images:", len(train_full_ds), " Test images:", len(test_ds))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'chest_xray_binary'

## 3) Dataloaders & Class Weights (Sanity Check Plot)

In [ ]:

# 3) Build split datasets and loaders + class weights
train_ds = datasets.ImageFolder(root=DATA_DIR, transform=tf_train)
val_ds   = datasets.ImageFolder(root=DATA_DIR, transform=tf_eval)
test_ds  = datasets.ImageFolder(root=DATA_DIR, transform=tf_eval)

from torch.utils.data import Subset
train_set = Subset(train_ds, train_idx)
val_set   = Subset(val_ds,   val_idx)
test_set  = Subset(test_ds,  test_idx)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Class weights
train_labels = targets[train_idx]
classes, counts = np.unique(train_labels, return_counts=True)
class_counts = {int(c): int(n) for c,n in zip(classes, counts)}
class_weights = torch.tensor([len(train_labels)/class_counts[c] for c in range(len(class_to_idx))], dtype=torch.float32).to(DEVICE)

# Save split stats
with open(os.path.join(OUT_DIR, "split_stats.json"), "w") as f:
    json.dump({
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "train_counts": class_counts,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "n_test": len(test_idx),
        "use_hist_equalize": USE_HIST_EQUALIZE
    }, f, indent=2)

# Quick sanity plot: class distribution in train
import matplotlib.pyplot as plt
plt.figure()
plt.bar([idx_to_class[i] for i in class_counts.keys()], [class_counts[i] for i in class_counts.keys()])
plt.title("Train Class Distribution")
plt.xlabel("Class"); plt.ylabel("Count")
plt.tight_layout()
plt.show()

print('Loaders ready. Class weights:', class_weights.tolist())


## 4) Model Definition (Transfer Learning)

In [ ]:

# 4) Model: ResNet18 transfer learning
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in model.parameters():
    p.requires_grad = False  # Phase A: head-only
num_feats = model.fc.in_features
model.fc = nn.Linear(num_feats, len(class_to_idx))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.fc.parameters(), lr=LR_HEAD)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

sum_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready. Trainable params: {sum_params} (head-only).')


## 5) Training & Evaluation Utilities

In [ ]:

# 5) Train/Eval utilities
def run_epoch(loader, train=False):
    model.train(train)
    total_loss = 0.0
    y_true, y_pred, y_prob = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * yb.size(0)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        y_true.extend(yb.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())
        y_prob.extend(probs.detach().cpu().numpy().tolist())
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(y_true), np.array(y_pred), np.array(y_prob)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

def compute_metrics(y_true, y_pred, y_prob, pos_index):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=pos_index, zero_division=0)
    try:
        auc = roc_auc_score((y_true==pos_index).astype(int), y_prob[:, pos_index])
    except Exception:
        auc = float('nan')
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc, "cm": cm}

def plot_confusion(cm, title):
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(class_to_idx))
    plt.xticks(tick_marks, [idx_to_class[i] for i in range(len(class_to_idx))], rotation=45)
    plt.yticks(tick_marks, [idx_to_class[i] for i in range(len(class_to_idx))])
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.ylabel('True label'); plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

print('Utilities ready.')


## 6) Phase A — Train Head-only (Early Stopping)

In [ ]:

# 6) Phase A: Train head-only with early stopping
pos_idx = class_to_idx[POSITIVE_CLASS_NAME]

history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1 = -1
epochs_no_improve = 0

for epoch in range(EPOCHS_PHASE_A):
    tr_loss, tr_y, tr_p, tr_prob = run_epoch(train_loader, train=True)
    va_loss, va_y, va_p, va_prob = run_epoch(val_loader, train=False)
    metrics = compute_metrics(va_y, va_p, va_prob, pos_idx)
    scheduler.step(va_loss)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["val_f1"].append(metrics["f1"])

    print(f"[A][{epoch+1}/{EPOCHS_PHASE_A}] "
          f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f} "
          f"val_acc={metrics['accuracy']:.3f} val_f1={metrics['f1']:.3f} val_rec={metrics['recall']:.3f} val_auc={metrics['roc_auc']:.3f}")

    improved = metrics["f1"] > best_val_f1 + 1e-4
    if improved:
        best_val_f1 = metrics["f1"]; epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(OUT_DIR, "best_resnet18_head.pth"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print("Early stopping (Phase A).")
            break

# Quick plots for sanity: loss & val F1
plt.figure()
plt.plot(history["train_loss"], label="train loss")
plt.plot(history["val_loss"], label="val loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Loss Curves"); plt.legend()
plt.tight_layout(); plt.show()

plt.figure()
plt.plot(history["val_f1"], label="val F1")
plt.xlabel("epoch"); plt.ylabel("F1"); plt.title("Validation F1"); plt.legend()
plt.tight_layout(); plt.show()

print('Phase A complete. Best val F1:', best_val_f1)


## 7) Phase B — Fine-tune Last Block (Optional)

In [ ]:

# 7) Phase B (optional): Fine-tune last block
best_head_path = os.path.join(OUT_DIR, "best_resnet18_head.pth")
if os.path.exists(best_head_path):
    model.load_state_dict(torch.load(best_head_path, map_location=DEVICE))

for p in model.layer4.parameters():
    p.requires_grad = True
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

best_val_f1_b = -1
epochs_no_improve = 0

for epoch in range(EPOCHS_PHASE_B):
    tr_loss, tr_y, tr_p, tr_prob = run_epoch(train_loader, train=True)
    va_loss, va_y, va_p, va_prob = run_epoch(val_loader, train=False)
    metrics = compute_metrics(va_y, va_p, va_prob, pos_idx)
    scheduler.step(va_loss)

    print(f"[B][{epoch+1}/{EPOCHS_PHASE_B}] "
          f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f} "
          f"val_acc={metrics['accuracy']:.3f} val_f1={metrics['f1']:.3f} val_rec={metrics['recall']:.3f} val_auc={metrics['roc_auc']:.3f}")

    improved = metrics["f1"] > best_val_f1_b + 1e-4
    if improved:
        best_val_f1_b = metrics["f1"]; epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(OUT_DIR, "best_resnet18_finetune.pth"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print("Early stopping (Phase B).")
            break

best_path = os.path.join(OUT_DIR, "best_resnet18_finetune.pth") if os.path.exists(os.path.join(OUT_DIR, "best_resnet18_finetune.pth")) else best_head_path
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))

print('Phase B complete. Loaded weights from:', best_path)


## 8) Baseline — Majority Class

In [ ]:

# 8) Majority-class baseline (required simple baseline)
train_counts = json.load(open(os.path.join(OUT_DIR, "split_stats.json")))["train_counts"]
majority_label = max(train_counts, key=lambda k: train_counts[str(k)] if isinstance(train_counts, dict) else train_counts[k]) if isinstance(train_counts, dict) else int(max(train_counts, key=lambda k: train_counts[k]))

# Evaluate baseline on test labels
discovery_labels = np.array([y for _, y in discovery_ds.samples])
test_targets = discovery_labels[test_idx]
baseline_preds = np.full_like(test_targets, fill_value=int(majority_label))

# dummy probs (for AUROC plotting convenience)
baseline_probs = np.zeros((len(test_targets), len(class_to_idx)))
baseline_probs[:, int(majority_label)] = 1.0

from sklearn.metrics import classification_report
baseline_metrics = {
    **{k:v for k,v in compute_metrics(test_targets, baseline_preds, baseline_probs, pos_idx).items() if k != "cm"},
}
print('Baseline metrics:', json.dumps({k:(float(v) if not isinstance(v, np.ndarray) else 'array') for k,v in baseline_metrics.items()}, indent=2))

# No confusion matrix plot here (all predictions identical).


## 9) Final Test Evaluation (CNN) & Reports

In [ ]:

# 9) Final Test Evaluation (CNN) + Confusion Matrix
te_loss, te_y, te_p, te_prob = run_epoch(test_loader, train=False)
test_metrics = compute_metrics(te_y, te_p, te_prob, pos_idx)

print("=== TEST RESULTS (CNN) ===")
print(json.dumps({k:(float(v) if not isinstance(v, np.ndarray) else 'array') for k,v in test_metrics.items() if k != "cm"}, indent=2))
print("Confusion matrix (rows=true, cols=pred):\n", test_metrics["cm"])

plot_confusion(test_metrics["cm"], "CNN Confusion Matrix (Test)")

# Save text classification reports (for appendix)
with open(os.path.join(OUT_DIR, "classification_report_test.txt"), "w") as f:
    f.write("=== CNN (Test) ===\n")
    f.write(classification_report(te_y, te_p, target_names=[idx_to_class[i] for i in range(len(class_to_idx))]))
    f.write("\n\n=== Majority Baseline (Test) ===\n")
    f.write(classification_report(test_targets, baseline_preds, target_names=[idx_to_class[i] for i in range(len(class_to_idx))]))

print('Saved classification_report_test.txt')


## 10) Artifacts Summary

In [ ]:

# 10) Artifacts summary (paths)
print(f"Artifacts directory: {OUT_DIR}")
for fn in ["split_stats.json", "best_resnet18_head.pth", "best_resnet18_finetune.pth", "classification_report_test.txt"]:
    p = os.path.join(OUT_DIR, fn)
    print(f"{fn}: {'exists' if os.path.exists(p) else 'missing'}")
